# Vault with identities and creds

`.env` holds only the link to a vault. Secrets are named `<platform>/<key>` and a
plugin says which it needs. Here KeePassXC, in a temporary file with a made-up
password, so nothing else is needed.

In [1]:
from amoorion import Amoorion

# We need its folder for the vault file.
# A sandbox for this whole notebook: secrets only from variables, the inbox and
# the dataset in a temporary folder. Nothing here can touch your real setup.
scratch = Amoorion.init()

In [2]:
# The library that reads KeePassXC files, only to look inside afterwards.
from pykeepass import PyKeePass

from amoorion import Plugins

# Each plugin says which secrets it needs.
from amoorion.plugins import load_plugins

from amoorion.vault import VaultLink

from amoorion.vault import kpxc_db_file

# A made-up password and a file in the sandbox: nothing real here.
filepath = scratch / "identity.kdbx"

# The link: which kind of vault, where, and its password.
link = VaultLink(
    kind="keepassxc",
    path=str(filepath),
    token="made-up",
)

# Make the file, with an empty entry per platform. `task vault:new` does this for real.
kpxc_db_file.initialize(
    link,
    {"ntfy": load_plugins().get("ntfy").schema()},
)

['/tmp/amoorion-gx_ozg0e/identity.kdbx',
 'group amoorion',
 'entry ntfy',
 'ntfy/topic',
 'ntfy/uri',
 'ntfy/token',
 'ntfy: notes']

That is what `task vault:new` does. Each entry's notes say what its fields are:

In [3]:
# Open it as KeePassXC itself would.
entry = PyKeePass(
    filepath,
    password="made-up",
).find_entries(
    title="ntfy",
    first=True,
)

# The entry explains, in its notes, which secret is what.
print(entry.notes)

One topic of an ntfy server

topic: The topic; on a public server anyone who knows it can read, so pick a long one
uri (optional): The server; default: https://ntfy.sh
token (optional): Access token, for protected topics


An identity is one env file pointing at one vault. Whether the secrets are there
shows as a flag, and values never:

In [4]:
# The env file is the only thing pointing at the vault.
env_file = scratch / "identity.env"

env_file.write_text(
    f"AMOORION_VAULT=keepassxc\nAMOORION_VAULT_PATH={filepath}\nAMOORION_VAULT_TOKEN=made-up\n"
)

# This object takes its secrets from that file.
amo = Amoorion(env_file)


# `accounts` asks the vault whether a platform has what it needs.
def ntfy_ready():

    return next(
        status["ready"]
        for status in amo.accounts()
        if status["platform"] == "ntfy"
    )


# The entry exists, yet it is empty: not ready.
print(
    ntfy_ready(),
    Plugins["ntfy"].has_secrets(amo.vault),
)

False False


In [5]:
db = PyKeePass(
    filepath,
    password="made-up",
)

# Fill in the secret, as you would in KeePassXC's window.
db.find_entries(
    title="ntfy",
    first=True,
).set_custom_property(
    "topic",
    "my-topic",
)

# Written to the file…
db.save()

# …and a new object sees it.
amo = Amoorion(env_file)

# Ready now.
print(
    ntfy_ready(),
    Plugins["ntfy"].has_secrets(amo.vault),
)

True True


In a terminal, `amoorion` alone asks everything with arrow menus: up and down choose, Enter confirms,
Esc or left goes back to the previous question. `amoorion accounts` is there too, to see which
platforms already have their secrets. Terminals only: a notebook has no keys to steer with.


`Amoorion("work.env")` and `Amoorion("private.env")` can live side by side:
`examples/advanced-usecases/scoping-of-identities-handling/`.